# NB4 — Treino e Avaliação LOSO por Contexto de Crise

**Pipeline de Predição de Crises Epilépticas a partir de EEG**

## Fluxo
```
NB3 → data/features/{dataset}__{pat}.npz  (X_pre, X_inter, t_pre, t_inter, ctx_ids)
NB2 → data/preictal_estimate.json          (PRE_SEC individual por paciente)
                        ↓
NB4 → 3 etapas sequenciais (1 variável livre por vez):
  Etapa 1: janela pré-ictal  (fixo: R5 + RF)       → W ótima por paciente
  Etapa 2: nível de canais   (fixo: W ótima + RF)   → nível ótimo global
  Etapa 3: modelo            (fixo: W ótima + nível ótimo) → melhor modelo
                        ↓
  data/results/stage{1,2,3}.csv  +  data/results/summary.csv
```

## Janelas experimentais (do NB2)
| Label | Duração | Origem |
|-------|---------|--------|
| W1 | 8 min | P25 da distribuição PRE_SEC |
| W2 | 10 min | P50 (mediana) |
| W3 | 13 min | P75 |
| W4 | 19 min | Máximo |
| W5 | PRE_SEC individual | PELT+KMeans do NB2 (fallback: mediana global 9,6 min) |

## Níveis de canal (fatiamento de colunas — NB3)
R5(17ch)→R4(15ch)→R3(13ch)→R2(11ch)→R1(4ch)→R0(2ch)
SeizeIT2: apenas R0 em todas as etapas.

## LOSO por contexto
- Cada contexto (par interictal+pré-ictal de uma crise) = 1 fold de teste
- Treino: contextos restantes com undersample de interictal 1:1
- Teste: distribuição natural (sem balanceamento)
- Métricas: AUC-ROC, F1, Sensibilidade, Especificidade, FP/h

In [1]:
import subprocess, sys
for pkg in ['scikit-learn','xgboost','tqdm']:
    try: __import__(pkg.replace('scikit-learn','sklearn'))
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
print('Dependências OK.')

Dependências OK.


In [2]:
import os, json, warnings, re
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, f1_score, recall_score, confusion_matrix
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False; print('XGBoost não instalado — pip install xgboost')

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Diretórios ────────────────────────────────────────────────────────────
ROOT       = Path('data')
FEAT_DIR   = ROOT / 'features'
RES_DIR    = ROOT / 'results'
RES_DIR.mkdir(parents=True, exist_ok=True)
FEAT_MAP   = ROOT / 'features_map.json'
PRE_EST    = ROOT / 'preictal_estimate.json'

# ── Janelas experimentais (derivadas do NB2) ──────────────────────────────
WINDOWS_FIXED = {'W1': 8*60, 'W2': 10*60, 'W3': 13*60, 'W4': 19*60}
GLOBAL_MEDIAN_S = 9.6 * 60   # fallback para W5 quando PRE_SEC=None
MAX_PRE_S = 40 * 60          # teto do segmento .npz

# ── Níveis de canal ───────────────────────────────────────────────────────
N_FEATS_CH = 19
LEVEL_COLS = {
    'R5': 17*N_FEATS_CH,  # 323
    'R4': 15*N_FEATS_CH,  # 285
    'R3': 13*N_FEATS_CH,  # 247
    'R2': 11*N_FEATS_CH,  # 209
    'R1':  4*N_FEATS_CH,  # 76
    'R0':  2*N_FEATS_CH,  # 38
}
STEP_SEG_S = 15.0   # passo de segmentação (para cálculo de FP/h)

# ── Modelos ───────────────────────────────────────────────────────────────
def make_rf():  return RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
def make_xgb(): return XGBClassifier(n_estimators=100, random_state=42,
                                      eval_metric='logloss', verbosity=0) if HAS_XGB else make_rf()
def make_lr():  return Pipeline([('scaler', StandardScaler()),
                                  ('clf', LogisticRegression(max_iter=1000, random_state=42, C=1.0))])
MODELS = {'RF': make_rf, 'XGB': make_xgb, 'LR': make_lr}

print('Configuração carregada.')
print(f'Janelas: {list(WINDOWS_FIXED.keys())} + W5 (individual)')
print(f'Níveis : {list(LEVEL_COLS.keys())}')
print(f'Modelos: {list(MODELS.keys())}')

c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuração carregada.
Janelas: ['W1', 'W2', 'W3', 'W4'] + W5 (individual)
Níveis : ['R5', 'R4', 'R3', 'R2', 'R1', 'R0']
Modelos: ['RF', 'XGB', 'LR']


In [3]:
# Carrega catálogo de features (gerado pelo NB3)
with open(FEAT_MAP, encoding='utf-8') as f:
    feat_map_raw = json.load(f)

# features_map pode ser lista ou dict com 'patients'
if isinstance(feat_map_raw, dict) and 'patients' in feat_map_raw:
    feat_patients = feat_map_raw['patients']
else:
    feat_patients = feat_map_raw

# Índice por (dataset, paciente)
PAT_MAP = {}  # (ds, pat) → meta dict
for entry in feat_patients:
    key = (entry['dataset'], entry['paciente'])
    PAT_MAP[key] = entry

# PRE_SEC individual por paciente (NB2)
PRE_SEC_PAT = {}
if PRE_EST.exists():
    with open(PRE_EST, encoding='utf-8') as f:
        pre_data = json.load(f)
    for r in pre_data:
        PRE_SEC_PAT[(r['dataset'], r['paciente'])] = r.get('pre_sec')
    print(f'PRE_SEC carregado para {len(PRE_SEC_PAT)} pacientes')
else:
    print('AVISO: preictal_estimate.json não encontrado — W5 usará mediana global')

def get_W5(ds, pat):
    """Retorna PRE_SEC em segundos, capped em MAX_PRE_S."""
    ps = PRE_SEC_PAT.get((ds, pat))
    if ps is None: ps = GLOBAL_MEDIAN_S
    return min(float(ps), MAX_PRE_S)

def get_all_windows(ds, pat):
    """Retorna dict {label: W_s} com as 5 janelas para este paciente."""
    w = dict(WINDOWS_FIXED)
    w['W5'] = get_W5(ds, pat)
    return w

print(f'\nPacientes no catálogo de features: {len(PAT_MAP)}')
for ds in ['CHBMIT','Siena','SeizeIT2','Mendeley']:
    n = sum(1 for (d,p) in PAT_MAP if d==ds)
    print(f'  {ds}: {n} pacientes')

PRE_SEC carregado para 24 pacientes

Pacientes no catálogo de features: 24
  CHBMIT: 7 pacientes
  Siena: 7 pacientes
  SeizeIT2: 7 pacientes
  Mendeley: 3 pacientes


## Função LOSO — núcleo do experimento

In [4]:
def run_loso_patient(ds, pat, W_s, level_cols, model_name, seed=42):
    """LOSO por contexto para um paciente × janela × nível × modelo.

    Returns pd.DataFrame com uma linha por fold, ou None se dados insuficientes.
    """
    np.random.seed(seed)
    entry = PAT_MAP.get((ds, pat))
    if entry is None: return None

    npz_path = Path(entry['path'])
    if not npz_path.exists(): return None

    # Carrega .npz
    d = np.load(npz_path, allow_pickle=True)
    meta = json.loads(str(d['meta']))

    avail_cols = d['X_pre'].shape[1]
    level_cols = min(level_cols, avail_cols)  # SeizeIT2 tem só 38 cols

    X_pre  = d['X_pre'][:, :level_cols].astype(np.float32)
    X_inter = d['X_inter'][:, :level_cols].astype(np.float32)
    t_pre  = d['t_pre']
    t_inter = d['t_inter']
    ctx_pre  = d['ctx_ids_pre']
    ctx_inter = d['ctx_ids_inter']

    # Filtra para a janela W
    inter_dur = float(t_inter.max())
    mask_p = t_pre   >= -W_s
    mask_i = t_inter >= (inter_dur - W_s)

    Xp = X_pre[mask_p];   cp = ctx_pre[mask_p]
    Xi = X_inter[mask_i]; ci = ctx_inter[mask_i]

    contexts = sorted(set(cp))
    if len(contexts) < 2: return None

    rows = []
    for test_ctx in contexts:
        # Split
        Xp_tr = Xp[cp != test_ctx]; Xi_tr = Xi[ci != test_ctx]
        Xp_te = Xp[cp == test_ctx]; Xi_te = Xi[ci == test_ctx]

        if len(Xp_tr) == 0 or len(Xi_tr) == 0: continue

        # Undersample interictal no treino → 1:1
        n_pre = len(Xp_tr)
        if len(Xi_tr) > n_pre:
            idx = np.random.choice(len(Xi_tr), n_pre, replace=False)
            Xi_tr = Xi_tr[idx]

        X_train = np.vstack([Xp_tr, Xi_tr])
        y_train = np.concatenate([np.ones(len(Xp_tr)), np.zeros(len(Xi_tr))])
        X_test  = np.vstack([Xp_te, Xi_te])
        y_test  = np.concatenate([np.ones(len(Xp_te)), np.zeros(len(Xi_te))])

        if len(set(y_train)) < 2 or len(set(y_test)) < 2: continue

        # NaN check
        if np.any(np.isnan(X_train)) or np.any(np.isnan(X_test)):
            X_train = np.nan_to_num(X_train)
            X_test  = np.nan_to_num(X_test)

        # Treina
        model = MODELS[model_name]()
        try:
            model.fit(X_train, y_train)
            if hasattr(model, 'predict_proba'):
                y_prob = model.predict_proba(X_test)[:, 1]
            elif hasattr(model, 'decision_function'):
                y_prob = model.decision_function(X_test)
                y_prob = (y_prob - y_prob.min()) / (y_prob.max() - y_prob.min() + 1e-10)
            else:
                y_prob = model.predict(X_test).astype(float)
        except Exception as e:
            continue

        y_pred = (y_prob >= 0.5).astype(int)

        # Métricas
        try:   auc = float(roc_auc_score(y_test, y_prob))
        except: auc = float('nan')

        f1   = float(f1_score(y_test, y_pred, zero_division=0))
        sens = float(recall_score(y_test, y_pred, pos_label=1, zero_division=0))
        try:
            tn,fp,fn,tp = confusion_matrix(y_test, y_pred, labels=[0,1]).ravel()
            spec = float(tn/(tn+fp)) if (tn+fp)>0 else float('nan')
        except: spec = float('nan')

        # FP/h: FP do interictal de teste / horas de monitorização
        h_inter_test = len(Xi_te) * STEP_SEG_S / 3600
        try:    fp_h = float(fp / h_inter_test) if h_inter_test > 0 else float('nan')
        except: fp_h = float('nan')

        rows.append({
            'dataset': ds, 'paciente': pat,
            'fold_ctx': int(test_ctx),
            'n_train_pre': len(Xp_tr), 'n_train_inter': len(Xi_tr),
            'n_test_pre':  len(Xp_te), 'n_test_inter':  len(Xi_te),
            'auc': auc, 'f1': f1,
            'sensitivity': sens, 'specificity': spec, 'fp_h': fp_h,
        })

    return pd.DataFrame(rows) if rows else None


def summarize_loso(df):
    """Agrega métricas de todos os folds de um paciente."""
    if df is None or len(df) == 0:
        return {'auc_mean': float('nan'), 'auc_std': float('nan'),
                'f1_mean': float('nan'),  'sens_mean': float('nan'),
                'spec_mean': float('nan'), 'fp_h_mean': float('nan'),
                'n_folds': 0}
    return {
        'auc_mean':  df['auc'].mean(),  'auc_std':   df['auc'].std(),
        'f1_mean':   df['f1'].mean(),   'sens_mean': df['sensitivity'].mean(),
        'spec_mean': df['specificity'].mean(), 'fp_h_mean': df['fp_h'].mean(),
        'n_folds':   len(df),
    }

print('Função LOSO definida.')

Função LOSO definida.


## Etapa 1 — Qual janela pré-ictal é melhor?

**Variável livre:** janela W ∈ {W1=8min, W2=10min, W3=13min, W4=19min, W5=PRE_SEC individual}
**Fixo:** nível R5 (17 canais × 19 features = 323 colunas), modelo Random Forest

Ao final: cada paciente tem sua W ótima (maior AUC média nos folds LOSO).

In [5]:
STAGE1_CSV = RES_DIR / 'stage1_windows.csv'
OVERWRITE_S1 = False

if STAGE1_CSV.exists() and not OVERWRITE_S1:
    df_s1 = pd.read_csv(STAGE1_CSV)
    print(f'Stage 1 carregado do CSV: {len(df_s1)} linhas')
else:
    MODEL_S1 = 'RF'
    LEVEL_S1 = 'R5'
    cols_S1  = LEVEL_COLS[LEVEL_S1]

    all_rows = []
    pbar = tqdm(sorted(PAT_MAP.keys()), desc='Etapa 1', unit='pac', colour='blue')

    for (ds, pat) in pbar:
        pbar.set_postfix_str(f'{ds}/{pat}')
        windows = get_all_windows(ds, pat)

        for wlabel, W_s in windows.items():
            df_folds = run_loso_patient(ds, pat, W_s, cols_S1, MODEL_S1)
            if df_folds is None or len(df_folds) == 0: continue
            for _, row in df_folds.iterrows():
                all_rows.append({
                    'stage':      1,
                    'dataset':    ds,      'paciente': pat,
                    'win_label':  wlabel,  'win_min':  round(W_s/60, 1),
                    'level':      LEVEL_S1, 'model':   MODEL_S1,
                    **{k: row[k] for k in ['fold_ctx','n_train_pre','n_train_inter',
                                            'n_test_pre','n_test_inter',
                                            'auc','f1','sensitivity','specificity','fp_h']},
                })

    df_s1 = pd.DataFrame(all_rows)
    df_s1.to_csv(STAGE1_CSV, index=False)
    print(f'Stage 1 concluído — {len(df_s1)} linhas → {STAGE1_CSV}')

Etapa 1: 100%|██████████| 24/24 [02:30<00:00,  6.28s/pac, Siena/PN17]      

Stage 1 concluído — 650 linhas → data\results\stage1_windows.csv


In [6]:
# ── Análise Stage 1: janela ótima por paciente ───────────────────────────
agg_s1 = (df_s1.groupby(['dataset','paciente','win_label','win_min'])
               .agg(auc_mean=('auc','mean'), auc_std=('auc','std'),
                    f1_mean=('f1','mean'), sens_mean=('sensitivity','mean'),
                    fp_h_mean=('fp_h','mean'), n_folds=('fold_ctx','count'))
               .reset_index())

# Janela ótima por paciente = maior AUC média (empate → menor janela)
best_win = (agg_s1.sort_values(['dataset','paciente','auc_mean','win_min'],
                                ascending=[True,True,False,True])
                  .groupby(['dataset','paciente']).first().reset_index()
                  [['dataset','paciente','win_label','win_min','auc_mean','auc_std']])

best_win.to_csv(RES_DIR / 'best_window_per_patient.csv', index=False)

print('JANELA ÓTIMA POR PACIENTE (Etapa 1)')
print('='*68)
print(f'  {"Dataset":<10} {"Paciente":<12} {"W ótima":<8} {"Min":<7} {"AUC médio":<12} {"±std"}')
print('  ' + '-'*65)
for _, r in best_win.iterrows():
    print(f'  {r.dataset:<10} {r.paciente:<12} {r.win_label:<8} {r.win_min:<7.1f} '
          f'{r.auc_mean:<12.3f} {r.auc_std:.3f}')

# Distribuição das janelas ótimas
print()
print('Distribuição das W ótimas:')
for wl, cnt in best_win['win_label'].value_counts().items():
    print(f'  {wl}: {cnt} pacientes')

# Janelas sugeridas para Etapa 2 (usa W ótima de cada paciente)
BEST_WIN_LOOKUP = {(r.dataset, r.paciente): r.win_label
                   for _, r in best_win.iterrows()}

JANELA ÓTIMA POR PACIENTE (Etapa 1)
  Dataset    Paciente     W ótima  Min     AUC médio    ±std
  -----------------------------------------------------------------
  CHBMIT     chb04        W3       13.0    0.841        0.068
  CHBMIT     chb06        W5       13.7    0.926        0.075
  CHBMIT     chb07        W4       19.0    0.775        0.054
  CHBMIT     chb08        W5       9.6     0.822        0.222
  CHBMIT     chb10        W4       19.0    0.955        0.030
  CHBMIT     chb13        W1       8.0     1.000        0.000
  CHBMIT     chb14        W4       19.0    0.821        0.172
  Mendeley   p10          W4       19.0    0.513        0.078
  Mendeley   p13          W2       10.0    0.946        0.073
  Mendeley   p15          W5       9.6     0.506        0.339
  SeizeIT2   sub-002      W4       19.0    0.803        0.277
  SeizeIT2   sub-034      W1       8.0     0.858        0.181
  SeizeIT2   sub-035      W2       10.0    0.926        0.168
  SeizeIT2   sub-039      W1 

## Etapa 2 — Qual nível de canais é melhor?

**Variável livre:** nível ∈ {R5, R4, R3, R2, R1, R0}
**Fixo:** W ótima individual de cada paciente (Etapa 1), modelo Random Forest
**SeizeIT2:** apenas R0 aplicável (2 canais behind-the-ear nativos).

In [7]:
STAGE2_CSV = RES_DIR / 'stage2_levels.csv'
OVERWRITE_S2 = False

if STAGE2_CSV.exists() and not OVERWRITE_S2:
    df_s2 = pd.read_csv(STAGE2_CSV)
    print(f'Stage 2 carregado do CSV: {len(df_s2)} linhas')
else:
    MODEL_S2 = 'RF'
    all_rows = []
    pbar = tqdm(sorted(PAT_MAP.keys()), desc='Etapa 2', unit='pac', colour='green')

    for (ds, pat) in pbar:
        pbar.set_postfix_str(f'{ds}/{pat}')
        entry     = PAT_MAP[(ds, pat)]
        is_seizeit = entry.get('is_seizeit2', ds == 'SeizeIT2')

        # W ótima desta etapa (da Etapa 1)
        wlabel = BEST_WIN_LOOKUP.get((ds, pat), 'W2')
        W_s    = get_all_windows(ds, pat).get(wlabel, 10*60)

        # Níveis a testar: SeizeIT2 só tem R0
        levels_to_test = ['R0'] if is_seizeit else list(LEVEL_COLS.keys())

        for level in levels_to_test:
            avail_cols = d_cols = LEVEL_COLS[level]
            df_folds = run_loso_patient(ds, pat, W_s, avail_cols, MODEL_S2)
            if df_folds is None or len(df_folds) == 0: continue
            for _, row in df_folds.iterrows():
                all_rows.append({
                    'stage': 2, 'dataset': ds, 'paciente': pat,
                    'win_label': wlabel, 'win_min': round(W_s/60, 1),
                    'level': level, 'model': MODEL_S2,
                    **{k: row[k] for k in ['fold_ctx','n_train_pre','n_train_inter',
                                            'n_test_pre','n_test_inter',
                                            'auc','f1','sensitivity','specificity','fp_h']},
                })

    df_s2 = pd.DataFrame(all_rows)
    df_s2.to_csv(STAGE2_CSV, index=False)
    print(f'Stage 2 concluído — {len(df_s2)} linhas → {STAGE2_CSV}')

Etapa 2: 100%|██████████| 24/24 [01:28<00:00,  3.68s/pac, Siena/PN17]      

Stage 2 concluído — 395 linhas → data\results\stage2_levels.csv


In [8]:
# ── Análise Stage 2: curva de degradação R5→R0 e nível ótimo global ─────
agg_s2 = (df_s2.groupby(['dataset','paciente','level'])
               .agg(auc_mean=('auc','mean'), auc_std=('auc','std'),
                    f1_mean=('f1','mean'), fp_h_mean=('fp_h','mean'),
                    n_folds=('fold_ctx','count'))
               .reset_index())

# Nível ótimo GLOBAL (excluindo SeizeIT2, que só tem R0)
non_sz = agg_s2[agg_s2['dataset'] != 'SeizeIT2']
level_global = (non_sz.groupby('level')['auc_mean'].mean()
                       .sort_values(ascending=False))

best_level_global = level_global.index[0]
BEST_LEVEL_LOOKUP = {(r.dataset, r.paciente): ('R0' if r.dataset=='SeizeIT2' else best_level_global)
                     for _, r in best_win.iterrows()}

print('CURVA DE DEGRADAÇÃO R5→R0 (Etapa 2 — média global excl. SeizeIT2)')
print('='*55)
level_order = ['R5','R4','R3','R2','R1','R0']
for lvl in level_order:
    if lvl in level_global.index:
        v = level_global[lvl]
        bar = '█' * int(v * 40)
        print(f'  {lvl}: {v:.3f}  {bar}')

print(f'\nNível ótimo global: {best_level_global}')
print(f'(SeizeIT2 usa sempre R0)')

CURVA DE DEGRADAÇÃO R5→R0 (Etapa 2 — média global excl. SeizeIT2)
  R5: 0.740  █████████████████████████████
  R4: 0.736  █████████████████████████████
  R3: 0.733  █████████████████████████████
  R2: 0.736  █████████████████████████████
  R1: 0.702  ████████████████████████████
  R0: 0.678  ███████████████████████████

Nível ótimo global: R5
(SeizeIT2 usa sempre R0)


## Etapa 3 — Qual modelo performa melhor?

**Variável livre:** modelo ∈ {RF, XGBoost, Logistic Regression}
**Fixo:** W ótima individual (Etapa 1), nível ótimo global (Etapa 2)
**SeizeIT2:** nível R0 independentemente do nível ótimo global.

In [9]:
STAGE3_CSV = RES_DIR / 'stage3_models.csv'
OVERWRITE_S3 = False

if STAGE3_CSV.exists() and not OVERWRITE_S3:
    df_s3 = pd.read_csv(STAGE3_CSV)
    print(f'Stage 3 carregado do CSV: {len(df_s3)} linhas')
else:
    all_rows = []
    pbar = tqdm(sorted(PAT_MAP.keys()), desc='Etapa 3', unit='pac', colour='yellow')

    for (ds, pat) in pbar:
        pbar.set_postfix_str(f'{ds}/{pat}')
        is_seizeit = PAT_MAP[(ds, pat)].get('is_seizeit2', ds == 'SeizeIT2')

        wlabel = BEST_WIN_LOOKUP.get((ds, pat), 'W2')
        W_s    = get_all_windows(ds, pat).get(wlabel, 10*60)
        level  = 'R0' if is_seizeit else best_level_global
        cols   = LEVEL_COLS[level]

        for model_name in MODELS:
            df_folds = run_loso_patient(ds, pat, W_s, cols, model_name)
            if df_folds is None or len(df_folds) == 0: continue
            for _, row in df_folds.iterrows():
                all_rows.append({
                    'stage': 3, 'dataset': ds, 'paciente': pat,
                    'win_label': wlabel, 'win_min': round(W_s/60, 1),
                    'level': level, 'model': model_name,
                    **{k: row[k] for k in ['fold_ctx','n_train_pre','n_train_inter',
                                            'n_test_pre','n_test_inter',
                                            'auc','f1','sensitivity','specificity','fp_h']},
                })

    df_s3 = pd.DataFrame(all_rows)
    df_s3.to_csv(STAGE3_CSV, index=False)
    print(f'Stage 3 concluído — {len(df_s3)} linhas → {STAGE3_CSV}')

Etapa 3: 100%|██████████| 24/24 [01:14<00:00,  3.09s/pac, Siena/PN17]      

Stage 3 concluído — 390 linhas → data\results\stage3_models.csv


In [10]:
# ── Análise Stage 3: comparação de modelos ───────────────────────────────
agg_s3 = (df_s3.groupby(['dataset','model'])
               .agg(auc_mean=('auc','mean'), auc_std=('auc','std'),
                    f1_mean=('f1','mean'), sens_mean=('sensitivity','mean'),
                    spec_mean=('specificity','mean'), fp_h_mean=('fp_h','mean'),
                    n_folds=('fold_ctx','count'))
               .reset_index())

print('COMPARAÇÃO DE MODELOS — AUC médio por dataset (Etapa 3)')
print('='*72)
print(f'  {"Dataset":<12} {"RF":<12} {"XGB":<12} {"LR":<12} {"Melhor"}')
print('  ' + '-'*68)
for ds in ['CHBMIT','Siena','Mendeley','SeizeIT2']:
    sub = agg_s3[agg_s3['dataset']==ds].set_index('model')
    vals = {m: sub.loc[m,'auc_mean'] if m in sub.index else float('nan')
            for m in ['RF','XGB','LR']}
    best = max(vals, key=lambda k: vals[k] if not pd.isna(vals[k]) else -1)
    print(f'  {ds:<12} {vals["RF"]:<12.3f} {vals["XGB"]:<12.3f} {vals["LR"]:<12.3f} {best}')
print()
# Global (todos datasets)
gbl = df_s3.groupby('model')['auc'].mean()
print('  GLOBAL:')
for m in ['RF','XGB','LR']:
    if m in gbl: print(f'    {m}: AUC = {gbl[m]:.3f}')

COMPARAÇÃO DE MODELOS — AUC médio por dataset (Etapa 3)
  Dataset      RF           XGB          LR           Melhor
  --------------------------------------------------------------------
  CHBMIT       0.888        0.880        0.825        RF
  Siena        0.645        0.641        0.579        RF
  Mendeley     0.634        0.570        0.688        LR
  SeizeIT2     0.901        0.890        0.800        RF

  GLOBAL:
    RF: AUC = 0.842
    XGB: AUC = 0.831
    LR: AUC = 0.763


## Resumo final — exportação completa

In [11]:
# Consolida os 3 stages num CSV master
all_dfs = []
for s, df in [(1,df_s1),(2,df_s2),(3,df_s3)]:
    if df is not None and len(df)>0: all_dfs.append(df)

if all_dfs:
    df_master = pd.concat(all_dfs, ignore_index=True)
    master_path = RES_DIR / 'resultados_completos.csv'
    df_master.to_csv(master_path, index=False)
    print(f'CSV master: {master_path}  ({len(df_master)} linhas)')

    # Tabela de summary por paciente × etapa (uma linha por configuração)
    summary_rows = []
    for s, df, fix_cols in [(1, df_s1, ['win_label','win_min']),
                             (2, df_s2, ['level']),
                             (3, df_s3, ['model'])]:
        if df is None: continue
        grp = df.groupby(['dataset','paciente'] + fix_cols).agg(
            auc_mean=('auc','mean'), auc_std=('auc','std'),
            f1_mean=('f1','mean'), sens_mean=('sensitivity','mean'),
            spec_mean=('specificity','mean'), fp_h_mean=('fp_h','mean'),
            n_folds=('fold_ctx','count')).reset_index()
        grp['stage'] = s
        summary_rows.append(grp)

    if summary_rows:
        df_summ = pd.concat(summary_rows, ignore_index=True)
        df_summ.to_csv(RES_DIR / 'summary_por_paciente.csv', index=False)
        print(f'Summary por paciente: {RES_DIR / "summary_por_paciente.csv"}')

print()
print('Arquivos gerados em', RES_DIR)
for f in sorted(RES_DIR.glob('*.csv')): print(f'  {f.name}')

CSV master: data\results\resultados_completos.csv  (1435 linhas)
Summary por paciente: data\results\summary_por_paciente.csv

Arquivos gerados em data\results
  best_window_per_patient.csv
  resultados_completos.csv
  stage1_windows.csv
  stage2_levels.csv
  stage3_models.csv
  summary_por_paciente.csv
